# Itinerary Generator Pipeline — Step-by-Step

Walks through `services/trip-generation/` one phase at a time. Run cells
top-to-bottom the first time; afterwards you can tweak any intermediate
variable and re-run just the downstream phase.

**Phases**
1. Route search & composition
2. POI retrieval + enrichment + union with route POIs
3. LLM pick (Gemini)
4. Solver — order stops, enforce opening hours & meal anchors
5. Persist (optional — writes to Supabase)

**Before you start:** see `notebooks/README.md` for tslab install. Launch
`jupyter lab` from the **repo root**, not from `notebooks/`.

## Setup — load `.env.local` and sanity-check keys

In [ ]:
import { resolve } from "node:path";
import { existsSync } from "node:fs";

// Assumes jupyter was started from the repo root.
const envPath = resolve(process.cwd(), ".env.local");
if (!existsSync(envPath)) {
  throw new Error(`Missing .env.local at ${envPath}. Did you start jupyter from the repo root?`);
}
process.loadEnvFile(envPath);

const required = [
  "NEXT_PUBLIC_SUPABASE_URL",
  "SUPABASE_SECRET_KEY",
  "GEMINI_API_KEY",
];
const missing = required.filter((k) => !process.env[k]);
if (missing.length) throw new Error(`Missing env vars: ${missing.join(", ")}`);

console.log("env loaded:", required.map((k) => `${k}=${process.env[k]?.slice(0, 8)}…`).join("  "));
console.log("GOOGLE_PLACES_API_KEY:", process.env.GOOGLE_PLACES_API_KEY ? "set" : "(missing — enrichment will be skipped)");


## Imports — pipeline functions

Path alias `@/*` is wired through `notebooks/tsconfig.json` so we can use
the same imports the production code does.

In [ ]:
import {
  searchRoutesByVibe,
  composeFromRoutes,
  type RouteCandidate,
  type RouteComposition,
} from "@/services/trip-generation/route-engine";
import {
  searchPoisByVibe,
  loadPoisByIds,
  enrichWithLiveData,
  type PoiCandidate,
  type EnrichedPoi,
} from "@/services/trip-generation/poi-engine";
import { solveItinerary, PACE_CAPS, type RoutedDay, type SolverResult } from "@/services/trip-generation/solver";
import { __notebook, type PickResult, type SolveTry } from "@/services/trip-generation/orchestrator";
import type { SurveyAnswers } from "@/services/trip-generation";
import type { GenerateInput } from "@/services/trip-generation/generator";
import { randomBytes } from "node:crypto";

const genId = randomBytes(4).toString("hex");
console.log("genId for this notebook run:", genId);


## Pre-flight — ensure seed data exists for the example destination

Phase 1a needs `route_templates` rows for the destination (with matching
pace + alias) and Phase 2b needs the route's `place_ids` to resolve in
`poi_embeddings` or the health filter drops them.

This cell idempotently seeds a small Hokkaido ski corpus (3 resorts +
3 single-day routes) so the rest of the notebook can run end-to-end on a
cold database. It skips work when the routes are already present, and
upserts are keyed on stable identifiers so re-running is safe.

If you've already run `scripts/ingest-ski-dataset.ts` +
`scripts/seed-route-templates.ts`, this cell becomes a no-op.

In [ ]:
import { createAdminClient } from "@/lib/db";
import { generateEmbedding as _genEmb } from "@/lib/gemini";
import { normalizeAliases } from "@/lib/destination-aliases";

const _SEED_DEST = "Hokkaido, Japan";
const _SEED_ALIASES = normalizeAliases([
  "Hokkaido, Japan", "Hokkaido", "北海道", "Japan", "Japan ski", "ski japan",
]);

const _SEED_RESORTS = [
  { id: "rusutsu", name: "Rusutsu Resort", nameZh: "留壽都", lat: 42.7375, lng: 140.9089,
    notes: "Hokkaido's largest single resort, world-renowned for powder.", onsen: true, night: true },
  { id: "kiroro-resort", name: "Kiroro Resort", nameZh: "KIRORO", lat: 43.0789, lng: 140.9531,
    notes: "Inland resort known for exceptional snow quality.", onsen: true, night: false },
  { id: "furano", name: "Furano Ski Resort", nameZh: "富良野", lat: 43.3464, lng: 142.3760,
    notes: "Operated by Prince Hotels. One of the steepest mountains in Hokkaido.", onsen: false, night: false },
];

const _db = createAdminClient();

// Skip the embedding calls if every expected route is already present.
const { data: _existingRoutes } = await _db
  .from("route_templates")
  .select("title")
  .eq("destination_name", _SEED_DEST)
  .eq("is_archived", false);
const _have = new Set((_existingRoutes ?? []).map((r) => r.title));
const _missing = _SEED_RESORTS.filter((r) => !_have.has(`${r.nameZh} 滑雪一日`));

if (_missing.length === 0) {
  console.log(`pre-flight: all ${_SEED_RESORTS.length} routes already present for ${_SEED_DEST}, skipping`);
} else {
  console.log(`pre-flight: seeding ${_missing.length} POI + route pair(s) for ${_SEED_DEST}`);
  for (const r of _missing) {
    const tags = [
      "ski", "snow", "winter",
      ...(r.onsen ? ["onsen", "relaxed"] : []),
      ...(r.night ? ["nightlife"] : []),
      "adventure",
    ];
    const periods: Array<{ openDay: number; openMinutes: number; closeDay: number; closeMinutes: number }> = [];
    for (let d = 0; d < 7; d++) {
      periods.push({ openDay: d, openMinutes: 8 * 60 + 30, closeDay: d, closeMinutes: 16 * 60 + 30 });
      if (r.night) {
        periods.push({ openDay: d, openMinutes: 17 * 60, closeDay: d, closeMinutes: 20 * 60 + 30 });
      }
    }

    const poiText = `Ski resort in Hokkaido, Japan. ${r.name} (${r.nameZh}). ${r.notes}`;
    const poiEmbedding = await _genEmb(poiText);
    const { error: poiErr } = await _db.from("poi_embeddings").upsert(
      {
        place_id: r.id,
        destination_name: _SEED_DEST,
        destination_aliases: _SEED_ALIASES,
        name: r.name,
        item_type: "activity",
        tags,
        description: r.notes,
        embedding: poiEmbedding,
        lat: r.lat,
        lng: r.lng,
        live_data: {
          placeId: r.id,
          name: r.name,
          address: null,
          rating: null,
          priceLevel: null,
          lat: r.lat,
          lng: r.lng,
          openingPeriods: periods,
        },
        source: "notebook-seed",
        last_seen_at: new Date().toISOString(),
      },
      { onConflict: "place_id" }
    );
    if (poiErr) throw new Error(`poi upsert failed for ${r.id}: ${poiErr.message}`);

    const routeTitle = `${r.nameZh} 滑雪一日`;
    const routeText = `Travel experiences in ${_SEED_DEST} for a balanced-paced trip with a ${tags.join(", ")} vibe. ${routeTitle}. ${r.notes}`;
    const routeEmbedding = await _genEmb(routeText);
    const { error: routeErr } = await _db.from("route_templates").upsert(
      {
        destination_name: _SEED_DEST,
        destination_aliases: _SEED_ALIASES,
        title: routeTitle,
        summary: r.notes,
        vibe_tags: tags,
        pace: "balanced",
        place_ids: [r.id],
        pinned_vibes: ["ski"],
        embedding: routeEmbedding,
        source: "notebook-seed",
        last_health_ok_at: new Date().toISOString(),
      },
      { onConflict: "destination_name,title", ignoreDuplicates: false }
    );
    if (routeErr) throw new Error(`route upsert failed for ${r.id}: ${routeErr.message}`);
    console.log(`  + ${r.id} → "${routeTitle}"`);
  }
  console.log("pre-flight: done");
}


## Phase 0 — Survey input

Edit this cell to change the destination, party, vibe, etc., then re-run
everything below. The defaults below match the pre-flight seed (Hokkaido
ski, balanced pace, ski/snow/onsen vibe) so Phase 1a clears the 0.72 gate
on a freshly seeded database. If you change the destination, make sure
the corresponding `route_templates` + `poi_embeddings` rows exist —
otherwise Phase 1a returns 0 and Phase 2a falls back to Google Places.

In [ ]:
const answers: SurveyAnswers = {
  destination: "Hokkaido, Japan",
  duration_days: 2,
  party: "couple",
  party_size: 2,
  budget_tier: "mid",
  vibe: ["ski", "snow", "onsen"],
  pace: "balanced",
  must_haves: null,
};

const input: GenerateInput = {
  answers,
  authorLineUserId: "U_notebook_dev",
  startDate: undefined, // defaults to today + 14d
};

__notebook.validateAnswers(input);
const startWeekday = __notebook.deriveStartWeekday(input.startDate);
console.log("validated. startWeekday =", startWeekday, " (0=Sun … 6=Sat)");


## Phase 1a — Search curated routes

Vector-search `route_templates` by vibe. Returns up to 10 candidate
single-day routes, scored on similarity + boost + quality + pinned vibes.
Gate is 0.72 — anything below is filtered out.

In [ ]:
const routes: RouteCandidate[] = await searchRoutesByVibe({
  destination: answers.destination!,
  vibe: answers.vibe,
  pace: answers.pace,
  budget: answers.budget_tier,
  k: 10,
  genId,
});

console.log(`routes found: ${routes.length}`);
console.table(routes.slice(0, 10).map((r) => ({
  routeId: r.routeId.slice(0, 8),
  title: r.title,
  score: r.finalScore.toFixed(3),
  similarity: r.similarity.toFixed(3),
  places: r.placeIds.length,
})));


## Phase 1b — Compose routes into day-slots

Greedy packer assigns routes to days, avoiding place_id collisions. Any
day that isn't covered drops into `uncoveredDays`, which Phase 3 (LLM)
will fill.

In [ ]:
const compose: RouteComposition = composeFromRoutes(routes, answers.duration_days!);

console.log("covered days:");
for (const [day, route] of compose.coveredDays) {
  console.log(`  D${day}  ${route.title}  (${route.placeIds.length} stops)`);
}
console.log("uncovered days (LLM will pick for these):", compose.uncoveredDays);
console.log("place_ids reserved by routes:", compose.usedPlaceIds.size);


## Phase 2a — Retrieve POI candidates

Vector ANN search on `poi_embeddings`. Falls back to Google Places text
search if the corpus is cold for this destination.

In [ ]:
const poiCandidates: PoiCandidate[] = await searchPoisByVibe({
  destination: answers.destination!,
  vibe: answers.vibe,
  pace: answers.pace,
  budget: answers.budget_tier,
  k: 30,
  genId,
});

console.log(`POI candidates: ${poiCandidates.length}`);
console.table(poiCandidates.slice(0, 15).map((p) => ({
  name: p.name,
  type: p.itemType,
  similarity: p.similarity.toFixed(3),
  tags: (p.tags ?? []).slice(0, 4).join(","),
})));


## Phase 2b — Union with route POIs

Routes lock specific place_ids; we materialize them as POI candidates
and merge into the shortlist. Route POIs take precedence.

In [ ]:
const routePois = await loadPoisByIds(Array.from(compose.usedPlaceIds), genId);
const allCandidates = __notebook.unionByPlaceId(routePois, poiCandidates);

console.log(`route POIs: ${routePois.length} | search POIs: ${poiCandidates.length} | unioned: ${allCandidates.length}`);
if (allCandidates.length === 0) throw new Error("no candidates — pipeline would abort with no_candidates");


## Phase 2c — Enrich with live data (Google Places)

Batch-fetches address, coords, and opening periods. The solver requires
coords + opening hours; without them, a POI is effectively unschedulable.

In [ ]:
const enriched: EnrichedPoi[] = await enrichWithLiveData(allCandidates);

const withCoords = enriched.filter((e) => e.lat != null && e.lng != null).length;
const withHours = enriched.filter((e) => (e.live?.openingPeriods?.length ?? 0) > 0).length;
console.log(`enriched: ${enriched.length} total | ${withCoords} with coords | ${withHours} with opening hours`);
console.table(enriched.slice(0, 10).map((e) => ({
  name: e.name,
  type: e.itemType,
  coords: e.lat != null ? `${e.lat.toFixed(3)},${e.lng!.toFixed(3)}` : "(none)",
  hours: e.live?.openingPeriods?.length ?? 0,
  address: e.live?.address?.slice(0, 40) ?? "(none)",
})));


## Phase 3 — LLM pick (Gemini)

Gemini sees only place_id, name, type, tags, and a short summary — never
coords or hours. It returns `{ title, summary, tags, days: [{day_number,
place_ids[]}] }`. The orchestrator filters hallucinated/duplicate IDs and
truncates to the pace cap before returning.

We only ask the LLM to cover days that routes didn't already claim.

In [ ]:
let pick: PickResult | null = null;

if (compose.uncoveredDays.length === 0) {
  console.log("all days route-covered — skipping LLM. Synthesizing title/summary from routes.");
  pick = __notebook.synthesizePickFromRoutes(compose, answers.destination!);
} else {
  pick = await __notebook.llmPickAssignment(
    input,
    enriched,
    [], // no prior infeasibility issues on first attempt
    undefined, // no prior pick
    {
      onlyDays: compose.uncoveredDays,
      excludePlaceIds: compose.usedPlaceIds,
      genId,
      attempt: 0,
    }
  );
}

console.log("title  :", pick.title);
console.log("summary:", pick.summary);
console.log("tags   :", pick.tags.join(", "));
const byId = new Map(enriched.map((p) => [p.placeId, p]));
for (const d of pick.days) {
  const names = d.place_ids.map((id) => byId.get(id)?.name ?? `(unknown ${id.slice(0,8)})`);
  console.log(`  D${d.day_number}: ${names.join(" → ")}`);
}


### (Optional) Hand-edit the LLM pick

If you want to override the assignment before the solver runs, mutate
`pick.days` here. Example: force a different place_id on day 2.

```ts
// pick!.days = pick!.days.map(d => d.day_number === 2
//   ? { ...d, place_ids: [enriched[0].placeId, enriched[3].placeId] }
//   : d
// );
```

## Phase 4 — Solver

Brute-force permutes each day's stops (≤6 → ≤720 permutations), simulates
travel via Haversine, enforces opening hours and meal anchors (lunch
12–14, dinner 18–20). Returns either `feasible` with timed stops or
`infeasible` with issues — in the real pipeline, infeasibility triggers a
repair loop (LLM swap + route demotion, ≤2 attempts).

In [ ]:
const solved: SolveTry = __notebook.trySolve(pick, enriched, input, startWeekday, compose);

if (solved.kind === "feasible") {
  console.log("FEASIBLE\n");
  for (const day of solved.days) {
    console.log(`Day ${day.dayNumber}`);
    for (const s of day.stops) {
      const arrive = `${String(Math.floor(s.arriveMinutes/60)).padStart(2,"0")}:${String(s.arriveMinutes%60).padStart(2,"0")}`;
      const depart = `${String(Math.floor(s.departMinutes/60)).padStart(2,"0")}:${String(s.departMinutes%60).padStart(2,"0")}`;
      console.log(`  ${arrive}–${depart}  ${s.poi.name}  [${s.poi.itemType}]`);
    }
    console.log("");
  }
} else {
  console.log("INFEASIBLE — issues:");
  for (const issue of solved.issues) {
    console.log(`  D${issue.dayNumber} / ${issue.reason}: ${issue.detail}`);
    if (issue.offendingPlaceIds.length) {
      const names = issue.offendingPlaceIds.map((id) => byId.get(id)?.name ?? id.slice(0,8));
      console.log(`    offending: ${names.join(", ")}`);
    }
  }
  console.log("\nTo simulate the repair loop, re-run Phase 3 with these issues passed in as the 3rd arg of llmPickAssignment, and `pick` as the 4th arg.");
}


### (Optional) Manual repair attempt

Uncomment to re-ask the LLM with the infeasibility report. The orchestrator
does this automatically up to `MAX_REPAIR_ATTEMPTS = 2` times.

```ts
// if (solved.kind === "infeasible" && pick) {
//   const repairedPick = await __notebook.llmPickAssignment(
//     input,
//     enriched,
//     solved.issues.filter((i) => !compose.coveredDays.has(i.dayNumber)),
//     pick,
//     { onlyDays: compose.uncoveredDays, excludePlaceIds: compose.usedPlaceIds, genId, attempt: 1 }
//   );
//   const resolved = __notebook.trySolve(repairedPick, enriched, input, startWeekday, compose);
//   console.log("after repair:", resolved.kind);
// }
```

## Phase 5 — Persist (commented out)

Writes a real `trip_templates` + `trip_template_versions` +
`trip_template_items` row to Supabase. **Uncomment only when you want a
real artifact.**

In [ ]:
// if (solved.kind === "feasible" && pick) {
//   const out = await __notebook.persistTemplate(input, pick, solved.days, enriched, compose);
//   console.log("persisted:", out);
// } else {
//   console.log("skipping persist — solver was not feasible.");
// }
